# VisionGuard — Phase C: finish (23L-0764)
T4 GPU, same secrets. Run all (~25 min). Rebuilds report assets that lived only in the expired Phase B runtime, picks the final champion, downloads `phaseC_results.zip`.

In [ ]:
# --- setup: same in both notebooks (Colab's own TensorFlow/Keras pair; pinning another Keras breaks fit() on TF 2.21) -------------------------------------------------
from google.colab import userdata
import os, subprocess, datetime
GH_USER = 'Anasnaveed69'
REPO = '23L-0764_A2_Ops'
U, T = userdata.get('DAGSHUB_USER'), userdata.get('DAGSHUB_TOKEN')   # Colab Secrets panel (key icon)
os.chdir('/content')
!rm -rf {REPO} && git clone -q https://github.com/{GH_USER}/{REPO}.git
os.chdir(f'/content/{REPO}')
!pip -q install "mlflow==2.22.5" keras-tuner==1.4.7 "dvc[s3]==3.67.1" python-dotenv -r requirements-serve.txt
!pip -q install -e . --no-deps
!git config user.name "Anas Naveed Butt" && git config user.email "{U}@users.noreply.dagshub.com"
# MLflow -> DagsHub, read from .env (never committed)
open('.env','w').write(f'''MLFLOW_TRACKING_URI=https://dagshub.com/{U}/{REPO}.mlflow
MLFLOW_TRACKING_USERNAME={U}
MLFLOW_TRACKING_PASSWORD={T}
''')
print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M'), '| DagsHub user:', U, '| repo:', REPO)
!git log --oneline -3

In [ ]:
os.environ['VG_CACHE_DIR'] = '/content/tfcache'   # decoded images cached on disk across runs
!dvc remote modify origin --local access_key_id {T} && dvc remote modify origin --local secret_access_key {T}
!dvc pull -q && find data/raw -type f | wc -l && wc -l data/splits/*.csv
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
!bash scripts/colab_regen.sh

In [ ]:
from google.colab import files; files.download('/content/phaseC_results.zip')